# N08 训练循环与数据管线

> **对应理论篇**：[`05-训练循环与数据管线.md`](../../01-基础/05-训练循环与数据管线.md)、[`06-初始化与学习率调度.md`](../../01-基础/06-初始化与学习率调度.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/pretrain.py`（训练循环/数据/调度）、`common.py:86` `get_lr_multiplier`
> **Colab**：CPU 可跑，约 1 分钟。

## 学习目标
1. 手写 `Dataset`/`DataLoader`，理解 block 采样与 `shuffle`；
2. 验证 next-token 标签的右移关系，理解 teacher forcing；
3. 实现 warmup+cosine 调度并与 nanoproof 的 warmup/warmdown 对照；
4. 串起「数据 → 前向 → 反向 → 优化 → 评估」完整循环。


In [1]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip -q install torch

In [2]:
import os, sys
def find_code_dir():
    for c in ["../../01-基础/code/from_scratch", "01-基础/code/from_scratch",
              "../01-基础/code/from_scratch", "code/from_scratch"]:
        if os.path.exists(os.path.join(c, "configs.py")):
            return os.path.abspath(c)
    cur = os.getcwd()
    for _ in range(8):
        p = os.path.join(cur, "01-基础", "code", "from_scratch")
        if os.path.exists(os.path.join(p, "configs.py")):
            return p
        cur = os.path.dirname(cur)
    raise FileNotFoundError("找不到 from_scratch 代码目录")
sys.path.insert(0, find_code_dir())


## 理论回顾

**next-token 数据集**：长文本切成 $T+1$ 的块，输入取前 $T$ 个、标签取后 $T$ 个：

$$
(x_1,\dots,x_T),\quad (x_2,\dots,x_{T+1})
$$

`DataLoader` 负责批量、打乱、多进程加载。

**学习率调度**：nanoproof 用 linear warmup + flat + linear warmdown
（`common.py:86`）；本 notebook 同时实现更常见的 warmup + cosine：

$$
\eta_t=\eta_{\min}+\tfrac12\big(1+\cos(\pi p_t)\big)(\eta_{\max}-\eta_{\min}),
\qquad p_t=\frac{t-T_{\text{warm}}}{T_{\max}-T_{\text{warm}}}
$$

**梯度累积**：$K$ 个 micro-batch 的梯度平均 = 大 batch 梯度，显存不变。


## 实验 1：从零实现 Dataset + DataLoader

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from data import load_text, CharTokenizer

text = load_text()
tok = CharTokenizer.from_text(text)
ids = torch.tensor(tok.encode(text), dtype=torch.long)
print("文本长度", len(ids), "| vocab", tok.vocab_size)

class BlockDataset(Dataset):
    def __init__(self, ids, block):
        self.ids, self.block = ids, block
    def __len__(self):
        return max(0, len(self.ids) - self.block)
    def __getitem__(self, i):
        c = self.ids[i:i + self.block + 1]
        return c[:-1], c[1:]        # x, y = 右移一位

block = 64
train = BlockDataset(ids, block)
x, y = train[0]
print("x[:16]:", tok.decode(x[:16].tolist()))
print("y 是 x 右移一位:", torch.equal(x[1:], y[:-1]))
loader = DataLoader(train, batch_size=8, shuffle=True, drop_last=True)
xb, yb = next(iter(loader))
print("batch:", xb.shape, yb.shape)


文本长度 1209 | vocab 56
x[:16]: theorem add_comm
y 是 x 右移一位: True
batch: torch.Size([8, 64]) torch.Size([8, 64])


## 实验 2：学习率调度（warmup+cosine vs nanoproof warmdown）

In [4]:
import math

def warmup_cosine(t, T_max, warm=20, lr=3e-3, min_lr=3e-4):
    if t < warm:
        return lr * (t + 1) / warm
    p = (t - warm) / max(1, T_max - warm)
    return min_lr + 0.5 * (1 + math.cos(math.pi * min(1.0, p))) * (lr - min_lr)

def nanoproof_lr(progress, warmup_ratio=0.1, warmdown_ratio=0.8, final=0.1):
    if warmup_ratio > 0 and progress < warmup_ratio:
        return (progress + 1e-8) / warmup_ratio
    if progress <= 1.0 - warmdown_ratio:
        return 1.0
    d = (progress - (1.0 - warmdown_ratio)) / warmdown_ratio
    return (1 - d) + d * final

T = 100
print("step | warmup+cosine(1e-3) | nanoproof mult")
for t in (0, 10, 20, 50, 90, 99):
    print(f"{t:4d} | {warmup_cosine(t, T):18.4f} | {nanoproof_lr(t/T):.3f}")


step | warmup+cosine(1e-3) | nanoproof mult
   0 |             0.0002 | 0.000
  10 |             0.0016 | 1.000
  20 |             0.0030 | 1.000
  50 |             0.0022 | 0.662
  90 |             0.0004 | 0.213
  99 |             0.0003 | 0.111


## 实验 3：完整训练循环（DataLoader + 评估）

In [5]:
import time, torch
from configs import get_config
from model import GPT

device = "cuda" if torch.cuda.is_available() else "cpu"
n_val = max(1, len(ids) // 10)
train_ds = BlockDataset(ids[:-n_val], block)
val_ds = BlockDataset(ids[-n_val:], block)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, drop_last=True)

cfg = get_config("micro"); cfg.vocab_size = tok.vocab_size; cfg.sequence_len = block
model = GPT(cfg).to(device)
opt = model.configure_optimizers(0.1, 3e-3, (0.9, 0.95), device)

def evaluate():
    model.eval(); total = 0.0; n = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            _, loss = model(xb.to(device), yb.to(device))
            total += loss.item(); n += 1
    model.train(); return total / max(1, n)

max_steps = 200; t0 = time.time()
step = 0
while step < max_steps:
    for xb, yb in train_loader:
        lr = warmup_cosine(step, max_steps)
        for g in opt.param_groups: g["lr"] = lr
        _, loss = model(xb.to(device), yb.to(device))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if step % 50 == 0:
            print(f"step {step:3d} train {loss.item():.4f} lr {lr:.2e}")
        step += 1
        if step >= max_steps: break
print(f"完成 {time.time()-t0:.1f}s | 最终 val loss {evaluate():.4f}")


step   0 train 4.0317 lr 1.50e-04


step  50 train 1.1319 lr 2.82e-03


step 100 train 0.3374 lr 1.88e-03


step 150 train 0.1645 lr 7.82e-04


完成 3.7s | 最终 val loss 3.6170


## 思考题

1. `shuffle=True` 对训练有什么作用？为什么验证集通常不 shuffle？
2. `drop_last=True` 在什么情况下会影响与梯度累积的搭配？
3. warmup 为什么能缓解训练初期的不稳定？
4. `DataLoader` 多进程（`num_workers>0`）在什么场景收益最大？

## 改动实验（任选 2–3 个）
1. 把 `block` 从 64 改成 128，比较每步耗时与 loss。
2. 把 `warmup_cosine` 换成 `nanoproof_lr` 的线性 warmdown，比较最终 loss。
3. 加 `num_workers=2`（Linux）观察吞吐。
4. 把 batch 32 拆成 batch 8 × 累积 4，验证 loss 轨迹与直接 batch 32 接近。

## 预期输出
```
文本长度 ... | vocab ...
y 是 x 右移一位: True
batch: torch.Size([8, 64]) torch.Size([8, 64])
step   0 train 3.7... lr ...
完成 ...s | 最终 val loss 1.x
```
